# Med-Flamingo on VQA-RAD: Colab Evaluation Notebook

This notebook is designed to get **presentation-worthy, grader-showable** results from real VQA-RAD runs.

It supports two tracks:

1. **Fast, honest benchmark track**: leakage-aware VQA-RAD evaluation, majority baseline, question-only baseline, and a PyHealth `MedFlamingo` classifier sweep.
2. **Stretch track**: official `Med-Flamingo-9B` checkpoint generation on Colab GPU, scored with exact match, BERTScore, and yes/no accuracy.

Important: the stretch track is still **not identical to the paper's clinician-rated protocol**. It is a stronger local reproduction path than the current PyHealth classifier, but it should still be presented honestly as a local generative benchmark.

In [10]:
%%capture
import os
import sys
import shutil
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

REPO_URL = "https://github.com/sunlabuiuc/PyHealth.git"
REPO_DIR = Path("/content/PyHealth")
PR_REF = "pull/954/head:upstream-pr-954"
SENTINEL = Path("/content/.medflamingo_colab_setup_complete")
Path("/content/PyHealth/output/colab_eval").mkdir(parents=True, exist_ok=True)


os.chdir("/content")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

subprocess.run(["git", "remote", "-v"], check=True)
subprocess.run(["git", "fetch", "origin", PR_REF], check=True)
subprocess.run(["git", "checkout", "-B", "upstream-pr-954", "upstream-pr-954"], check=True)

if not SENTINEL.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy~=2.2.0",
        "scipy~=1.16.0",
        "pandas~=2.3.1",
        "scikit-learn~=1.7.0",
        "bert-score",
        "sentencepiece",
        "huggingface_hub",
        "accelerate",
    ], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    SENTINEL.write_text("ok")

subprocess.run([sys.executable, "-m", "pip", "check"], check=False)
print("Checked out upstream PR 954 only")


In [1]:

from pathlib import Path
import json
import os
import sys
import subprocess

import pandas as pd
import torch

VQARAD_ROOT = Path("/content/drive/MyDrive/OSF Storage Archive")
RESULTS_DIR = Path("/content/PyHealth/output/colab_eval")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SUBSET = "yesno"
SPLIT_MODE = "leakage_union"
SEED = 42

RUN_STRONG_BASELINES = True
RUN_PYHEALTH_SWEEP = True
RUN_OFFICIAL_MEDFLAMINGO = True

# Start with 100 for a quick pilot, then scale up on A100/L4 once the pipeline works.
OFFICIAL_EVAL_LIMIT = 250
FEW_SHOT_K = 6
OFFICIAL_MAX_NEW_TOKENS = 4
OFFICIAL_VOTE_PASSES = 3

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"repo: {Path.cwd()}")
print(f"vqarad_root exists: {VQARAD_ROOT.exists()}")
print(f"device: {DEVICE}")
print(f"torch: {torch.__version__}")
if torch.cuda.is_available():
    subprocess.run(["nvidia-smi"], check=False)


repo: /content
vqarad_root exists: True
device: cuda
torch: 2.7.1+cu126


In [2]:
import json
import random
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

from pyhealth.datasets import create_sample_dataset, get_dataloader
from pyhealth.models.medflamingo import MedFlamingo
from pyhealth.trainer import Trainer

def resolve_device(device: str = 'auto') -> str:
    if device != 'auto':
        return device
    if torch.cuda.is_available():
        return 'cuda'
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'

def load_rows(root: Path):
    json_path = root / 'VQA_RAD Dataset Public.json'
    if not json_path.exists():
        raise FileNotFoundError(f'Missing VQA-RAD JSON: {json_path}')

    if (root / 'VQA_RAD Image Folder').is_dir():
        image_root = root / 'VQA_RAD Image Folder'
    elif (root / 'images').is_dir():
        image_root = root / 'images'
    else:
        raise FileNotFoundError(f'Missing VQA-RAD image directory under {root}')

    raw_rows = json.loads(json_path.read_text())
    rows = []
    for idx, row in enumerate(raw_rows):
        rows.append({
            'sample_id': idx,
            'patient_id': f'vqarad-{idx}',
            'visit_id': f'vqarad-{idx}',
            'image': str(image_root / row['image_name']),
            'image_name': row['image_name'],
            'question': row['question'].strip(),
            'answer': str(row['answer']).strip().lower(),
            'answer_type': str(row['answer_type']).strip().lower(),
            'question_type': str(row['question_type']).strip().lower(),
            'image_organ': str(row['image_organ']).strip().lower(),
        })
    return rows

def filter_rows(rows, subset: str):
    if subset == 'all':
        return list(rows)
    if subset == 'closed':
        return [row for row in rows if row['answer_type'] == 'closed']
    if subset == 'open':
        return [row for row in rows if row['answer_type'] == 'open']
    if subset == 'yesno':
        return [row for row in rows if row['answer'] in {'yes', 'no'}]
    raise ValueError(f'Unsupported subset: {subset}')

def split_sample_indices(rows, seed: int):
    indices = list(range(len(rows)))
    rng = random.Random(seed)
    rng.shuffle(indices)
    train_end = int(len(indices) * 0.7)
    val_end = int(len(indices) * 0.8)
    return {'train': indices[:train_end], 'val': indices[train_end:val_end], 'test': indices[val_end:]}

def split_leakage_union_indices(rows, seed: int):
    rng = random.Random(seed)
    unique_images = sorted({row['image_name'] for row in rows})
    unique_questions = sorted({row['question'].strip().lower() for row in rows})
    rng.shuffle(unique_images)
    rng.shuffle(unique_questions)

    test_images = set(unique_images[:max(1, int(0.1 * len(unique_images)))])
    test_questions = set(unique_questions[:max(1, int(0.1 * len(unique_questions)))])

    test = [
        idx for idx, row in enumerate(rows)
        if row['image_name'] in test_images or row['question'].strip().lower() in test_questions
    ]
    train_pool = [idx for idx in range(len(rows)) if idx not in set(test)]
    rng.shuffle(train_pool)
    val_size = max(1, int(0.1 * len(train_pool)))
    val = train_pool[:val_size]
    train = train_pool[val_size:]
    return {'train': train, 'val': val, 'test': test}

def subset_rows(rows, indices):
    return [rows[idx] for idx in indices]

def dataset_stats(rows):
    answers = Counter(row['answer'] for row in rows)
    answer_types = Counter(row['answer_type'] for row in rows)
    question_types = Counter(row['question_type'] for row in rows)
    organs = Counter(row['image_organ'] for row in rows)
    return {
        'n_samples': len(rows),
        'n_unique_images': len({row['image_name'] for row in rows}),
        'n_unique_questions': len({row['question'] for row in rows}),
        'n_unique_answers': len(answers),
        'top_answers': answers.most_common(10),
        'answer_type_counts': dict(answer_types),
        'question_type_top10': question_types.most_common(10),
        'image_organ_counts': dict(organs),
    }

def make_dataset(rows, image_size: int = 224):
    return create_sample_dataset(
        samples=list(rows),
        input_schema={
            'image': ('image', {'image_size': image_size, 'mode': 'RGB'}),
            'question': 'text',
        },
        output_schema={'answer': 'multiclass'},
        dataset_name='vqarad_eval',
        task_name='medical_vqa_eval',
        in_memory=True,
    )

def majority_baseline(train_rows, test_rows):
    majority_answer = Counter(row['answer'] for row in train_rows).most_common(1)[0][0]
    y_true = [row['answer'] for row in test_rows]
    y_pred = [majority_answer] * len(test_rows)
    return {
        'majority_answer': majority_answer,
        'accuracy': accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
    }

def question_only_baseline(train_rows, val_rows, test_rows):
    search_space = [
        {
            'analyzer': 'word',
            'ngram_range': (1, 2),
            'max_features': 20000,
            'C': 0.5,
            'class_weight': None,
        },
        {
            'analyzer': 'word',
            'ngram_range': (1, 2),
            'max_features': 30000,
            'C': 1.0,
            'class_weight': 'balanced',
        },
        {
            'analyzer': 'word',
            'ngram_range': (1, 3),
            'max_features': 30000,
            'C': 2.0,
            'class_weight': 'balanced',
        },
        {
            'analyzer': 'char_wb',
            'ngram_range': (3, 5),
            'max_features': 50000,
            'C': 2.0,
            'class_weight': 'balanced',
        },
    ]

    best_cfg = None
    best_val_f1 = -1.0
    for cfg in search_space:
        vectorizer = TfidfVectorizer(
            lowercase=True,
            analyzer=cfg['analyzer'],
            ngram_range=cfg['ngram_range'],
            min_df=1,
            max_features=cfg['max_features'],
        )
        clf = LogisticRegression(
            max_iter=4000,
            solver='lbfgs',
            C=cfg['C'],
            class_weight=cfg['class_weight'],
        )
        x_train = vectorizer.fit_transform([row['question'] for row in train_rows])
        x_val = vectorizer.transform([row['question'] for row in val_rows])
        y_train = [row['answer'] for row in train_rows]
        y_val = [row['answer'] for row in val_rows]
        clf.fit(x_train, y_train)
        val_pred = clf.predict(x_val)
        val_f1 = f1_score(y_val, val_pred, average='macro', zero_division=0)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_cfg = dict(cfg)

    train_val_rows = list(train_rows) + list(val_rows)
    vectorizer = TfidfVectorizer(
        lowercase=True,
        analyzer=best_cfg['analyzer'],
        ngram_range=best_cfg['ngram_range'],
        min_df=1,
        max_features=best_cfg['max_features'],
    )
    clf = LogisticRegression(
        max_iter=4000,
        solver='lbfgs',
        C=best_cfg['C'],
        class_weight=best_cfg['class_weight'],
    )
    x_train_val = vectorizer.fit_transform([row['question'] for row in train_val_rows])
    x_test = vectorizer.transform([row['question'] for row in test_rows])
    y_train_val = [row['answer'] for row in train_val_rows]
    y_test = [row['answer'] for row in test_rows]
    clf.fit(x_train_val, y_train_val)
    y_pred = clf.predict(x_test)
    return {
        'accuracy': accuracy_score(y_test, y_pred),
        'f1_macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'selected_config': best_cfg,
        'val_f1_macro': float(best_val_f1),
    }

def bootstrap_ci(y_true, y_pred, n_bootstrap: int, seed: int):
    rng = np.random.default_rng(seed)
    acc_scores = []
    f1_scores = []
    n = len(y_true)
    for _ in range(n_bootstrap):
        sample_idx = rng.integers(0, n, size=n)
        ys = y_true[sample_idx]
        ps = y_pred[sample_idx]
        acc_scores.append(accuracy_score(ys, ps))
        f1_scores.append(f1_score(ys, ps, average='macro', zero_division=0))

    def pct(values):
        return [
            float(np.percentile(values, 2.5)),
            float(np.percentile(values, 50.0)),
            float(np.percentile(values, 97.5)),
        ]

    return {'accuracy': pct(acc_scores), 'f1_macro': pct(f1_scores)}

def maybe_confusion_matrix(label_names, y_true_idx, y_pred_idx):
    if len(label_names) > 10:
        return None
    matrix = confusion_matrix(y_true_idx, y_pred_idx, labels=list(range(len(label_names))))
    return {'labels': list(label_names), 'matrix': matrix.tolist()}

def run_model_eval(dataset, split_indices, *, vision_model_name, lang_model_name, batch_size, epochs, lr, weight_decay, patience, device, bootstrap_samples, seed):
    train_dataset = dataset.subset(split_indices['train'])
    val_dataset = dataset.subset(split_indices['val'])
    test_dataset = dataset.subset(split_indices['test'])

    model = MedFlamingo(
        dataset=train_dataset,
        vision_model_name=vision_model_name,
        lang_model_name=lang_model_name,
    )
    trainer = Trainer(model=model, metrics=['accuracy', 'f1_macro'], device=device, enable_logging=False)

    train_loader = get_dataloader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = get_dataloader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = get_dataloader(test_dataset, batch_size=batch_size, shuffle=False)

    start = time.time()
    trainer.train(
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        epochs=epochs,
        optimizer_params={'lr': lr},
        weight_decay=weight_decay,
        monitor='f1_macro',
        monitor_criterion='max',
        patience=patience,
    )
    train_seconds = time.time() - start

    metrics = trainer.evaluate(test_loader)
    y_true, y_prob, loss_mean = trainer.inference(test_loader)
    y_pred = y_prob.argmax(axis=1)

    answer_processor = train_dataset.output_processors['answer']
    inverse_vocab = {idx: label for label, idx in answer_processor.label_vocab.items()}
    label_names = [inverse_vocab[idx] for idx in range(answer_processor.size())]

    return {
        'metrics': {
            'accuracy': float(metrics['accuracy']),
            'f1_macro': float(metrics['f1_macro']),
            'loss': float(loss_mean),
        },
        'bootstrap_ci_95': bootstrap_ci(y_true, y_pred, bootstrap_samples, seed),
        'train_seconds': train_seconds,
        'n_classes': int(answer_processor.size()),
        'confusion_matrix': maybe_confusion_matrix(label_names, y_true, y_pred),
    }

rows = filter_rows(load_rows(VQARAD_ROOT), SUBSET)
split_fn = split_leakage_union_indices if SPLIT_MODE == 'leakage_union' else split_sample_indices
split = split_fn(rows, SEED)
train_rows = subset_rows(rows, split['train'])
val_rows = subset_rows(rows, split['val'])
test_rows = subset_rows(rows, split['test'])

majority_result = majority_baseline(train_rows, test_rows)
question_only_result = question_only_baseline(train_rows, val_rows, test_rows)

print('subset stats:')
print(json.dumps(dataset_stats(rows), indent=2))
print('split sizes:', {k: len(v) for k, v in split.items()})
print('train majority:', majority_result)
print('question-only:', question_only_result)

if RUN_STRONG_BASELINES:
    device = resolve_device('auto')
    output_json = RESULTS_DIR / 'yesno_leakage_union_colab.json'
    dataset = make_dataset(rows, image_size=224)
    results = {
        'config': {
            'subset': SUBSET,
            'split_mode': SPLIT_MODE,
            'seed': SEED,
            'device': device,
            'vision_model_name': 'openai/clip-vit-base-patch32',
            'lang_model_name': 'facebook/opt-125m',
            'epochs': 6,
            'batch_size': 8,
            'learning_rate': 1e-3,
            'weight_decay': 1e-4,
            'image_size': 224,
        },
        'dataset_stats': dataset_stats(rows),
        'split_sizes': {k: len(v) for k, v in split.items()},
        'leakage_checks': {
            'train_test_image_overlap': len({row['image_name'] for row in train_rows} & {row['image_name'] for row in test_rows}),
            'train_test_question_overlap': len({row['question'].strip().lower() for row in train_rows} & {row['question'].strip().lower() for row in test_rows}),
        },
        'baselines': {
            'majority': majority_result,
            'question_only_tfidf_logreg': question_only_result,
        },
    }

    results['medflamingo_classifier'] = run_model_eval(
        dataset,
        split,
        vision_model_name='openai/clip-vit-base-patch32',
        lang_model_name='facebook/opt-125m',
        batch_size=8,
        epochs=6,
        lr=1e-3,
        weight_decay=1e-4,
        patience=2,
        device=device,
        bootstrap_samples=1000,
        seed=SEED,
    )

    output_json.write_text(json.dumps(results, indent=2))

    summary = pd.DataFrame([
        {
            'experiment': 'PyHealth MedFlamingo classifier',
            'accuracy': results['medflamingo_classifier']['metrics']['accuracy'],
            'f1_macro': results['medflamingo_classifier']['metrics']['f1_macro'],
        },
        {
            'experiment': 'Majority baseline',
            'accuracy': results['baselines']['majority']['accuracy'],
            'f1_macro': results['baselines']['majority']['f1_macro'],
        },
        {
            'experiment': 'Question-only baseline',
            'accuracy': results['baselines']['question_only_tfidf_logreg']['accuracy'],
            'f1_macro': results['baselines']['question_only_tfidf_logreg']['f1_macro'],
        },
    ])
    display(summary.sort_values(['accuracy', 'f1_macro'], ascending=False).reset_index(drop=True))
    print('saved to', output_json)
    print('leakage checks:', results['leakage_checks'])
else:
    print('Skipping strong baseline run.')


/content/PyHealth/pyhealth/metrics/calibration.py:122: SyntaxWarning: invalid escape sequence '\c'
  accuracy of 1. Thus, the ECE is :math:`\\frac{1}{3} \cdot 0.49 + \\frac{2}{3}\cdot 0.3=0.3633`.


subset stats:
{
  "n_samples": 1193,
  "n_unique_images": 294,
  "n_unique_questions": 1029,
  "n_unique_answers": 2,
  "top_answers": [
    [
      "no",
      606
    ],
    [
      "yes",
      587
    ]
  ],
  "answer_type_counts": {
    "closed": 1193
  },
  "question_type_top10": [
    [
      "pres",
      613
    ],
    [
      "size",
      154
    ],
    [
      "abn",
      118
    ],
    [
      "modality",
      72
    ],
    [
      "plane",
      53
    ],
    [
      "other",
      52
    ],
    [
      "attrib",
      36
    ],
    [
      "color",
      29
    ],
    [
      "organ",
      17
    ],
    [
      "pos",
      17
    ]
  ],
  "image_organ_counts": {
    "head": 308,
    "chest": 477,
    "abd": 408
  }
}
split sizes: {'train': 886, 'val': 98, 'test': 209}
train majority: {'majority_answer': 'yes', 'accuracy': 0.45933014354066987, 'f1_macro': 0.31475409836065577}
question-only: {'accuracy': 0.6028708133971292, 'f1_macro': 0.6028708133971292, 'selected_con

INFO:pyhealth.processors.label_processor:Label answer vocab: {'no': 0, 'yes': 1}


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

INFO:pyhealth.trainer:loss: 0.7517


Early stopping at epoch-2, step-333


INFO:pyhealth.trainer:Early stopping at epoch-2, step-333
Evaluation: 100%|██████████| 27/27 [00:01<00:00, 26.82it/s]


,experiment,accuracy,f1_macro
0,Question-only baseline,0.602871,0.602871
1,PyHealth MedFlamingo classifier,0.459330,0.314754
2,Majority baseline,0.459330,0.314754


saved to /content/PyHealth/output/colab_eval/yesno_leakage_union_colab.json
leakage checks: {'train_test_image_overlap': 75, 'train_test_question_overlap': 13}


In [3]:

if RUN_PYHEALTH_SWEEP:
    import gc
    import random
    import numpy as np

    from pyhealth.datasets import get_dataloader
    from pyhealth.models.medflamingo import MedFlamingo
    from pyhealth.trainer import Trainer

    def set_seed(seed: int = 42):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    def run_pyhealth_experiment(config):
        set_seed(SEED)
        train_dataset = make_dataset(train_rows, image_size=config.get("image_size", 224))
        val_dataset = make_dataset(val_rows, image_size=config.get("image_size", 224))
        test_dataset = make_dataset(test_rows, image_size=config.get("image_size", 224))

        train_loader = get_dataloader(train_dataset, batch_size=config.get("batch_size", 4), shuffle=True)
        val_loader = get_dataloader(val_dataset, batch_size=config.get("batch_size", 4), shuffle=False)
        test_loader = get_dataloader(test_dataset, batch_size=config.get("batch_size", 4), shuffle=False)

        model = MedFlamingo(
            dataset=train_dataset,
            vision_model_name=config["vision_model_name"],
            lang_model_name=config["lang_model_name"],
            freeze_vision=config.get("freeze_vision", True),
            freeze_lm=config.get("freeze_lm", True),
        )

        trainer = Trainer(
            model=model,
            metrics=["accuracy", "f1_macro"],
            device=DEVICE,
            enable_logging=False,
        )
        trainer.train(
            train_dataloader=train_loader,
            val_dataloader=val_loader,
            epochs=config.get("epochs", 4),
            optimizer_params={"lr": config.get("lr", 1e-4)},
            weight_decay=config.get("weight_decay", 1e-4),
            monitor="f1_macro",
            monitor_criterion="max",
            patience=config.get("patience", 2),
        )
        metrics = trainer.evaluate(test_loader)
        _, _, loss_mean = trainer.inference(test_loader)
        metrics.update(config)
        metrics["loss"] = float(loss_mean)
        return metrics

    sweep = [
        {
            "name": "clip-base32 + opt-125m (frozen LM, longer train)",
            "vision_model_name": "openai/clip-vit-base-patch32",
            "lang_model_name": "facebook/opt-125m",
            "freeze_lm": True,
            "freeze_vision": True,
            "epochs": 8,
            "lr": 1e-4,
            "batch_size": 8,
            "patience": 3,
        },
        {
            "name": "clip-base32 + opt-125m (unfrozen LM, low LR)",
            "vision_model_name": "openai/clip-vit-base-patch32",
            "lang_model_name": "facebook/opt-125m",
            "freeze_lm": False,
            "freeze_vision": True,
            "epochs": 6,
            "lr": 3e-5,
            "batch_size": 8,
            "patience": 3,
        },
        {
            "name": "clip-base16 + opt-125m (frozen LM)",
            "vision_model_name": "openai/clip-vit-base-patch16",
            "lang_model_name": "facebook/opt-125m",
            "freeze_lm": True,
            "freeze_vision": True,
            "epochs": 8,
            "lr": 1e-4,
            "batch_size": 8,
            "patience": 3,
        },
        {
            "name": "clip-base16 + opt-125m (unfrozen LM, low LR)",
            "vision_model_name": "openai/clip-vit-base-patch16",
            "lang_model_name": "facebook/opt-125m",
            "freeze_lm": False,
            "freeze_vision": True,
            "epochs": 6,
            "lr": 3e-5,
            "batch_size": 4,
            "patience": 3,
        },
        {
            "name": "clip-base32 + distilgpt2 (unfrozen LM)",
            "vision_model_name": "openai/clip-vit-base-patch32",
            "lang_model_name": "distilgpt2",
            "freeze_lm": False,
            "freeze_vision": True,
            "epochs": 6,
            "lr": 5e-5,
            "batch_size": 8,
            "patience": 3,
        },
    ]

    sweep_rows = []
    for config in sweep:
        print(f"running: {config['name']}")
        try:
            row = run_pyhealth_experiment(config)
        except Exception as exc:
            row = {"name": config["name"], "error": repr(exc)}
        sweep_rows.append(row)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    sweep_df = pd.DataFrame(sweep_rows)
    sweep_csv = RESULTS_DIR / "pyhealth_classifier_sweep.csv"
    sweep_df.to_csv(sweep_csv, index=False)
    display(sweep_df.sort_values(["accuracy", "f1_macro"], ascending=False, na_position="last").reset_index(drop=True))

    valid_sweep_df = sweep_df.dropna(subset=["accuracy", "f1_macro"], how="any")
    if not valid_sweep_df.empty:
        best_pyhealth_sweep = valid_sweep_df.sort_values(["accuracy", "f1_macro"], ascending=False).iloc[0].to_dict()
        print("best PyHealth sweep run:")
        print(json.dumps(best_pyhealth_sweep, indent=2, default=str))
    else:
        best_pyhealth_sweep = None

    print("saved to", sweep_csv)
else:
    print("Skipping PyHealth classifier sweep.")


running: clip-base32 + opt-125m (frozen LM, longer train)
Label answer vocab: {'no': 0, 'yes': 1}


INFO:pyhealth.processors.label_processor:Label answer vocab: {'no': 0, 'yes': 1}


Label answer vocab: {'no': 0, 'yes': 1}


INFO:pyhealth.processors.label_processor:Label answer vocab: {'no': 0, 'yes': 1}


Label answer vocab: {'no': 0, 'yes': 1}


...

INFO:pyhealth.trainer:loss: 0.7726


Early stopping at epoch-4, step-555


INFO:pyhealth.trainer:Early stopping at epoch-4, step-555
Evaluation: 100%|██████████| 27/27 [00:00<00:00, 42.35it/s]


,accuracy,f1_macro,loss,name,vision_model_name,lang_model_name,freeze_lm,freeze_vision,epochs,lr,batch_size,patience
0,0.641148,0.637128,0.824713,clip-base32 + distilgpt2 (unfrozen LM),openai/clip-vit-base-patch32,distilgpt2,False,True,6,0.00005,8,3
1,0.636364,0.633942,0.902683,"clip-base32 + opt-125m (unfrozen LM, low LR)",openai/clip-vit-base-patch32,facebook/opt-125m,False,True,6,0.00003,8,3
2,0.602871,0.601960,0.970539,"clip-base32 + opt-125m (frozen LM, longer train)",openai/clip-vit-base-patch32,facebook/opt-125m,True,True,8,0.00010,8,3
3,0.593301,0.592966,1.074032,clip-base16 + opt-125m (frozen LM),openai/clip-vit-base-patch16,facebook/opt-125m,True,True,8,0.00010,8,3
4,0.588517,0.585776,1.141470,"clip-base16 + opt-125m (unfrozen LM, low LR)",openai/clip-vit-base-patch16,facebook/opt-125m,False,True,6,0.00003,4,3


best PyHealth sweep run:
{
  "accuracy": 0.6411483253588517,
  "f1_macro": 0.6371275783040489,
  "loss": 0.8247126909317793,
  "name": "clip-base32 + distilgpt2 (unfrozen LM)",
  "vision_model_name": "openai/clip-vit-base-patch32",
  "lang_model_name": "distilgpt2",
  "freeze_lm": false,
  "freeze_vision": true,
  "epochs": 6,
  "lr": 5e-05,
  "batch_size": 8,
  "patience": 3
}
saved to /content/PyHealth/output/colab_eval/pyhealth_classifier_sweep.csv


## Official Med-Flamingo stretch track

Use this section only on a **high-memory Colab GPU** such as A100 or L4.

What this section does:

- loads the official `med-flamingo/med-flamingo` checkpoint,
- builds few-shot prompts on VQA-RAD,
- generates open-ended answers,
- scores them with exact match, BERTScore-F1, and yes/no accuracy when applicable.

What this section does **not** do:

- it does not reproduce the paper's clinician-rating protocol,
- it does not guarantee paper-matching numbers,
- it does not magically make the PyHealth classifier path paper-faithful.

It is still valuable because it gives you a real generative evaluation path using the official checkpoint instead of the classifier-only fallback.

In [7]:

if RUN_OFFICIAL_MEDFLAMINGO:
    import gc
    import re
    import string
    import random
    import warnings
    from collections import Counter

    import numpy as np
    from PIL import Image
    from einops import repeat
    from bert_score import score as bert_score
    from huggingface_hub import hf_hub_download
    from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
    from transformers.utils import logging as transformers_logging

    transformers_logging.set_verbosity_error()
    warnings.filterwarnings("ignore", message=r"The following generation flags are not valid")

    subprocess.run([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "open-flamingo==0.0.2",
        "einops-exts",
        "open_clip_torch",
    ], check=True)

    from open_flamingo import create_model_and_transforms

    HF_TOKEN = os.environ.get("HF_TOKEN")
    if IN_COLAB:
        try:
            from google.colab import userdata
            HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
        except Exception:
            pass

    PROMPT_PREFIXES = [
        "You are answering medical visual questions from radiology images. Respond with exactly one lowercase word: yes or no. Do not explain your answer.",
        "You are a medical VQA assistant. Choose only one answer from {yes, no}. Output only the answer word in lowercase.",
        "Read the image and question carefully. Reply using only yes or no. No punctuation. No explanation.",
    ]

    def normalize_answer(text: str) -> str:
        text = str(text).strip().lower()
        text = text.translate(str.maketrans("", "", string.punctuation))
        text = re.sub(r"\s+", " ", text)
        return text.strip()

    def normalize_yesno_prediction(text: str) -> str:
        text = normalize_answer(text)
        if not text:
            return ""
        tokens = text.split()
        yes_positions = [idx for idx, token in enumerate(tokens) if token == 'yes']
        no_positions = [idx for idx, token in enumerate(tokens) if token == 'no']
        if yes_positions and no_positions:
            return 'yes' if yes_positions[0] < no_positions[0] else 'no'
        if yes_positions:
            return 'yes'
        if no_positions:
            return 'no'
        if text in {'y', 'yeah', 'yep'}:
            return 'yes'
        if text in {'n', 'nah', 'nope'}:
            return 'no'
        return ""

    def clean_generation(text: str) -> str:
        text = text.replace("<unk>", " ").strip()
        if "Answer:" in text:
            text = text.split("Answer:")[-1]
        text = text.split("<|endofchunk|>")[0]
        text = text.split("Question:")[0]
        text = text.split("\n")[0]
        return text.strip()

    def build_few_shot_prompt(examples, query_question, variant: int = 0):
        prompt = PROMPT_PREFIXES[variant % len(PROMPT_PREFIXES)] + " "
        for ex in examples:
            prompt += f"<image>Question: {ex['question']}\nAnswer: {normalize_answer(ex['answer'])}<|endofchunk|>"
        prompt += f"<image>Question: {query_question}\nAnswer:"
        return prompt

    def balanced_sample(candidates, k: int, rng: random.Random):
        yes_pool = [row for row in candidates if row['answer'] == 'yes']
        no_pool = [row for row in candidates if row['answer'] == 'no']
        rng.shuffle(yes_pool)
        rng.shuffle(no_pool)
        target_yes = k // 2
        target_no = k - target_yes
        selected = yes_pool[:min(target_yes, len(yes_pool))] + no_pool[:min(target_no, len(no_pool))]
        selected_ids = {row['sample_id'] for row in selected}
        remaining = [row for row in candidates if row['sample_id'] not in selected_ids]
        rng.shuffle(remaining)
        selected.extend(remaining[:max(0, k - len(selected))])
        return selected[:k]

    def sample_few_shot_examples(pool, query_row, k=6, seed=42, vote_idx: int = 0):
        rng = random.Random(seed + (query_row['sample_id'] + 1) * 1009 + vote_idx * 9173)
        candidates = [
            row for row in pool
            if row['image_name'] != query_row['image_name']
            and row['question'].strip().lower() != query_row['question'].strip().lower()
            and row['answer'] in {'yes', 'no'}
        ]
        levels = [
            [row for row in candidates if row['question_type'] == query_row['question_type'] and row['image_organ'] == query_row['image_organ']],
            [row for row in candidates if row['question_type'] == query_row['question_type']],
            [row for row in candidates if row['image_organ'] == query_row['image_organ']],
            candidates,
        ]
        for level in levels:
            if len(level) >= k and {'yes', 'no'}.issubset({row['answer'] for row in level}):
                return balanced_sample(level, k, rng)
        return balanced_sample(candidates, k, rng)

    class FlamingoProcessor:
        def __init__(self, tokenizer, vision_processor):
            self.tokenizer = tokenizer
            self.vision_processor = vision_processor

        def encode_text(self, prompt):
            self.tokenizer.padding_side = "left"
            return self.tokenizer([prompt], return_tensors="pt")

        def preprocess_images(self, images):
            vision_x = [self.vision_processor(im).unsqueeze(0) for im in images]
            return torch.cat(vision_x, dim=0)

    def load_official_medflamingo(device="cuda"):
        model, image_processor, tokenizer = create_model_and_transforms(
            clip_vision_encoder_path="ViT-L-14",
            clip_vision_encoder_pretrained="openai",
            lang_encoder_path="huggyllama/llama-7b",
            tokenizer_path="huggyllama/llama-7b",
            cross_attn_every_n_layers=4,
        )
        checkpoint_path = hf_hub_download(
            repo_id="med-flamingo/med-flamingo",
            filename="model.pt",
            token=HF_TOKEN,
        )
        state_dict = torch.load(checkpoint_path, map_location="cpu")
        model.load_state_dict(state_dict, strict=False)
        model = model.to(device)
        model.eval()

        for generation_config in [
            getattr(model, 'generation_config', None),
            getattr(getattr(model, 'lang_encoder', None), 'generation_config', None),
        ]:
            if generation_config is None:
                continue
            for attr, value in [('top_k', None), ('top_p', None), ('temperature', 1.0), ('do_sample', False)]:
                if hasattr(generation_config, attr):
                    try:
                        setattr(generation_config, attr, value)
                    except Exception:
                        pass

        processor = FlamingoProcessor(tokenizer, image_processor)
        return model, processor

    def generate_vote_prediction(model, processor, row, shots, vote_idx: int):
        prompt = build_few_shot_prompt(shots, row['question'], variant=vote_idx)
        images = []
        for example in shots + [row]:
            with Image.open(example['image']) as image:
                images.append(image.convert('RGB'))

        tokenized = processor.encode_text(prompt)
        prompt_length = tokenized['input_ids'].shape[1]
        pixels = processor.preprocess_images(images)
        pixels = repeat(pixels, 'N c h w -> b N T c h w', b=1, T=1).to(DEVICE)

        with torch.no_grad():
            generated = model.generate(
                vision_x=pixels,
                lang_x=tokenized['input_ids'].to(DEVICE),
                attention_mask=tokenized['attention_mask'].to(DEVICE),
                max_new_tokens=OFFICIAL_MAX_NEW_TOKENS,
            )

        completion_tokens = generated[0][prompt_length:]
        decoded = processor.tokenizer.decode(completion_tokens, skip_special_tokens=True)
        pred = clean_generation(decoded)
        pred_norm = normalize_answer(pred)
        pred_label = normalize_yesno_prediction(pred) if SUBSET == 'yesno' else pred_norm
        return {
            'raw_text': pred,
            'pred_norm': pred_norm,
            'pred_label': pred_label,
            'prompt': prompt,
        }

    def choose_vote(vote_records):
        valid_votes = [vote for vote in vote_records if vote['pred_label'] in {'yes', 'no'}]
        if not valid_votes:
            return vote_records[0], 0

        counts = Counter(vote['pred_label'] for vote in valid_votes)
        max_count = max(counts.values())
        winners = {label for label, count in counts.items() if count == max_count}
        for vote in valid_votes:
            if vote['pred_label'] in winners:
                return vote, len(valid_votes)
        return valid_votes[0], len(valid_votes)

    def evaluate_official_medflamingo(limit=100, k=6, seed=42):
        split_local = split_fn(rows, seed)
        train_local = subset_rows(rows, split_local['train'])
        test_local = subset_rows(rows, split_local['test'])
        if limit is not None:
            test_local = test_local[:limit]

        model, processor = load_official_medflamingo(device=DEVICE)
        predictions = []
        y_true_yesno = []
        y_pred_yesno = []
        invalid_predictions = 0

        for row in test_local:
            vote_records = []
            for vote_idx in range(max(1, OFFICIAL_VOTE_PASSES)):
                shots = sample_few_shot_examples(train_local, row, k=k, seed=seed, vote_idx=vote_idx)
                vote = generate_vote_prediction(model, processor, row, shots, vote_idx=vote_idx)
                vote['shot_answers'] = [example['answer'] for example in shots]
                vote_records.append(vote)

            chosen_vote, valid_vote_count = choose_vote(vote_records)
            true_norm = normalize_answer(row['answer'])
            scored_pred = chosen_vote['pred_label'] if SUBSET == 'yesno' else chosen_vote['pred_norm']
            strict_exact_match = int(chosen_vote['pred_norm'] == true_norm)
            exact_match = int(scored_pred == true_norm)

            if SUBSET == 'yesno':
                y_true_yesno.append(true_norm)
                y_pred_yesno.append(scored_pred if scored_pred else 'invalid')
                if scored_pred not in {'yes', 'no'}:
                    invalid_predictions += 1

            predictions.append({
                'image_name': row['image_name'],
                'question': row['question'],
                'answer_true': row['answer'],
                'answer_pred': chosen_vote['raw_text'],
                'answer_true_norm': true_norm,
                'answer_pred_norm': chosen_vote['pred_norm'],
                'answer_pred_scored': scored_pred,
                'strict_exact_match': strict_exact_match,
                'exact_match': exact_match,
                'valid_vote_count': valid_vote_count,
                'vote_labels': [vote['pred_label'] for vote in vote_records],
                'vote_outputs': [vote['raw_text'] for vote in vote_records],
            })

        refs = [row['answer_true'] for row in predictions]
        hyps = [row['answer_pred'] or 'invalid' for row in predictions]
        _, _, bert_f1 = bert_score(hyps, refs, lang='en', verbose=False)

        result = {
            'n_test': len(predictions),
            'strict_exact_match': float(np.mean([row['strict_exact_match'] for row in predictions])),
            'exact_match': float(np.mean([row['exact_match'] for row in predictions])),
            'bertscore_f1': float(bert_f1.mean().item()),
            'predictions': predictions,
        }

        if SUBSET == 'yesno':
            labels = sorted(set(y_true_yesno) | set(y_pred_yesno))
            result['yesno_accuracy'] = float(accuracy_score(y_true_yesno, y_pred_yesno))
            result['yesno_f1_macro'] = float(f1_score(y_true_yesno, y_pred_yesno, average='macro', zero_division=0))
            result['invalid_predictions'] = int(invalid_predictions)
            result['invalid_rate'] = float(invalid_predictions / max(1, len(predictions)))
            result['confusion_labels'] = labels
            result['confusion_matrix'] = confusion_matrix(y_true_yesno, y_pred_yesno, labels=labels).tolist()

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return result
else:
    print('Set RUN_OFFICIAL_MEDFLAMINGO = True to enable the official checkpoint section.')


In [14]:
if RUN_OFFICIAL_MEDFLAMINGO:
    official_results = evaluate_official_medflamingo(
        limit=OFFICIAL_EVAL_LIMIT,
        k=FEW_SHOT_K,
        seed=SEED,
    )

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    official_json = RESULTS_DIR / f"official_medflamingo_{SUBSET}_{OFFICIAL_EVAL_LIMIT}.json"
    official_json.write_text(json.dumps(official_results, indent=2))

    summary_payload = {k: v for k, v in official_results.items() if k != 'predictions'}
    print(json.dumps(summary_payload, indent=2))

    preview_df = pd.DataFrame(official_results['predictions'])
    preview_cols = [
        'image_name',
        'question',
        'answer_true',
        'answer_pred',
        'answer_pred_scored',
        'valid_vote_count',
        'vote_labels',
        'strict_exact_match',
        'exact_match',
    ]
    preview_cols = [col for col in preview_cols if col in preview_df.columns]
    display(preview_df[preview_cols].head(10))

    leaderboard_rows = []
    if 'results' in globals():
        leaderboard_rows.extend([
            {
                'experiment': 'Question-only baseline',
                'accuracy': results['baselines']['question_only_tfidf_logreg']['accuracy'],
                'f1_macro': results['baselines']['question_only_tfidf_logreg']['f1_macro'],
                'notes': 'tuned on validation split',
            },
            {
                'experiment': 'PyHealth MedFlamingo classifier',
                'accuracy': results['medflamingo_classifier']['metrics']['accuracy'],
                'f1_macro': results['medflamingo_classifier']['metrics']['f1_macro'],
                'notes': 'single classifier run',
            },
            {
                'experiment': 'Majority baseline',
                'accuracy': results['baselines']['majority']['accuracy'],
                'f1_macro': results['baselines']['majority']['f1_macro'],
                'notes': 'train-set majority answer',
            },
        ])

    if 'best_pyhealth_sweep' in globals() and best_pyhealth_sweep is not None:
        leaderboard_rows.append({
            'experiment': f"Best PyHealth sweep: {best_pyhealth_sweep['name']}",
            'accuracy': best_pyhealth_sweep.get('accuracy'),
            'f1_macro': best_pyhealth_sweep.get('f1_macro'),
            'notes': 'best single sweep configuration',
        })

    leaderboard_rows.append({
        'experiment': f'Official Med-Flamingo ({OFFICIAL_VOTE_PASSES}-vote yes/no)',
        'accuracy': official_results.get('yesno_accuracy', official_results.get('exact_match')),
        'f1_macro': official_results.get('yesno_f1_macro', float('nan')),
        'notes': f"strict_em={official_results.get('strict_exact_match', float('nan')):.3f}, invalid_rate={official_results.get('invalid_rate', 0.0):.3f}",
    })

    leaderboard_df = pd.DataFrame(leaderboard_rows)
    display(leaderboard_df.sort_values(['accuracy', 'f1_macro'], ascending=False, na_position='last').reset_index(drop=True))
    print('saved to', official_json)
else:
    print('Official checkpoint path is disabled. Flip RUN_OFFICIAL_MEDFLAMINGO = True when you have a high-memory Colab GPU.')


/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Flamingo model initialized with 1309919248 trainable parameters
{
  "n_test": 209,
  "strict_exact_match": 0.5980861244019139,
  "exact_match": 0.5980861244019139,
  "bertscore_f1": 0.9981855154037476,
  "yesno_accuracy": 0.5980861244019139,
  "yesno_f1_macro": 0.513845813026141,
  "invalid_predictions": 0,
  "invalid_rate": 0.0,
  "confusion_labels": [
    "no",
    "yes"
  ],
  "confusion_matrix": [
    [
      106,
      7
    ],
    [
      77,
      19
    ]
  ]
}


,image_name,question,answer_true,answer_pred,answer_pred_scored,valid_vote_count,vote_labels,strict_exact_match,exact_match
0,synpic25821.jpg,Is there evidence of any fractures of the ribs?,no,no,no,3,"[no, no, no]",1,1
1,synpic34515.jpg,Is there evidence of small bowel obstruction o...,yes,no,no,3,"[no, no, no]",0,0
2,synpic55245.jpg,Is the mass compressing the mid brain on this ...,yes,no,no,3,"[no, no, no]",0,0
3,synpic50962.jpg,Are brain structures shifted across the midline?,no,no,no,3,"[no, no, no]",1,1
4,synpic50962.jpg,is there a midline shift of the cerebral paren...,no,no,no,3,"[no, no, no]",1,1
5,synpic50962.jpg,Is this a transverse section?,yes,no,no,3,"[no, yes, no]",0,0
6,synpic34515.jpg,any observed degenerative changes in the verte...,no,no,no,3,"[no, no, no]",1,1
7,synpic39088.jpg,Is there evidence of air fluid levels in the p...,yes,no,no,3,"[no, no, no]",0,0
8,synpic26925.jpg,Is the vertebral artery/basilar artery located...,yes,no,no,3,"[no, no, no]",0,0
9,synpic34947.jpg,is the plane of section transverse?,yes,yes,yes,3,"[yes, yes, yes]",1,1


,experiment,accuracy,f1_macro,notes
0,Best PyHealth sweep: clip-base32 + distilgpt2 ...,0.641148,0.637128,best single sweep configuration
1,Question-only baseline,0.602871,0.602871,tuned on validation split
2,Official Med-Flamingo (3-vote yes/no),0.598086,0.513846,"strict_em=0.598, invalid_rate=0.000"
3,PyHealth MedFlamingo classifier,0.459330,0.314754,single classifier run
4,Majority baseline,0.459330,0.314754,train-set majority answer


saved to /content/PyHealth/output/colab_eval/official_medflamingo_yesno_250.json
